# Google Play Store AnalysisExploratory data analysis of the Google Play Store apps dataset and user reviews dataset, covering data cleaning, category and rating trends, pricing and revenue patterns, and sentiment analysis of user reviews.**Dataset source:** [Google Play Store Apps — Kaggle](https://www.kaggle.com/datasets/lava18/google-play-store-apps)

## 1. Load and Clean the Apps DatasetSteps performed:- Load `googleplaystore.csv`- Fix a known corrupted row (missing `Category` shifted every later column)- Remove duplicate rows and duplicate app listings (keeping the version with the highest review count)- Handle missing values in `Type`, `Current Ver`, `Android Ver`- Convert `Installs`, `Price`, and `Size` from text into proper numeric types

In [ ]:
import pandas as pdimport matplotlib.pyplot as plt# Loadapps = pd.read_csv('googleplaystore.csv')# Fix corrupted row (index 10472 has a missing Category, shifting all later columns)apps = apps[apps['Category'] != '1.9']# Remove exact duplicate rowsapps = apps.drop_duplicates()# Clean Reviews and remove duplicate app entries, keeping the row with the highest review countapps['Reviews'] = apps['Reviews'].astype(int)apps = apps.sort_values('Reviews', ascending=False).drop_duplicates(subset='App', keep='first')# Handle missing valuesapps = apps.dropna(subset=['Type'])apps['Current Ver'] = apps['Current Ver'].fillna('Unknown')apps['Android Ver'] = apps['Android Ver'].fillna('Unknown')# Clean Installs: "10,000+" -> 10000apps['Installs'] = apps['Installs'].str.replace(',', '').str.replace('+', '').astype(int)# Clean Price: "$4.99" -> 4.99apps['Price'] = apps['Price'].str.replace('$', '', regex=False).astype(float)# Clean Size: "19M" -> 19.0, "512k" -> 0.5, "Varies with device" -> NaNdef clean_size(x):    if pd.isna(x):        return None    if isinstance(x, str):        if x == 'Varies with device':            return None        if x.endswith('M'):            return float(x.replace('M', ''))        if x.endswith('k'):            return float(x.replace('k', '')) / 1024    return xapps['Size'] = apps['Size'].apply(clean_size)print("Final shape:", apps.shape)print(apps.isnull().sum())apps.head()

*Note: `Rating` and `Size` retain missing values intentionally — these represent apps with no ratings yet, or apps marked 'Varies with device', rather than data errors. Filling them would fabricate information we don't have.

## 2. Category AnalysisWhich categories have the most apps, and which actually get the most installs?

In [ ]:
apps['Category'].value_counts().head(10).plot(kind='bar', figsize=(10,5), color='skyblue')plt.title('Top 10 App Categories by Count')plt.xlabel('Category')plt.ylabel('Number of Apps')plt.xticks(rotation=45)plt.tight_layout()plt.show()

In [ ]:
apps.groupby('Category')['Installs'].sum().sort_values(ascending=False).head(10)

**Observation:** `FAMILY` has by far the most listed apps, but `GAME` and `COMMUNICATION` dominate total installs. App count in a category doesn't equal actual usage — a category can be saturated with apps that individually get very little traction.

## 3. Ratings Analysis

In [ ]:
apps['Rating'].describe()

In [ ]:
plt.figure(figsize=(8,5))apps['Rating'].hist(bins=20, color='mediumpurple', edgecolor='black')plt.title('Distribution of App Ratings')plt.xlabel('Rating')plt.ylabel('Number of Apps')plt.show()

In [ ]:
apps.groupby('Category')['Rating'].mean().sort_values(ascending=False).head(10)

In [ ]:
apps.groupby('Type')['Rating'].describe()

**Observation:** Ratings are left-skewed — median (4.3) is higher than the mean (4.17), meaning most apps cluster in the 4.0-4.5 range, with a smaller tail of poorly-rated apps pulling the average down. Free and paid apps show only a small ratings difference (paid apps average slightly higher, ~4.26 vs ~4.17).

## 4. Size vs Installs

In [ ]:
plt.figure(figsize=(8,5))plt.scatter(apps['Size'], apps['Installs'], alpha=0.3, color='teal')plt.title('App Size vs Installs')plt.xlabel('Size (MB)')plt.ylabel('Installs')plt.yscale('log')  # installs range is huge, log scale makes the pattern visibleplt.show()

In [ ]:
apps[['Size', 'Installs']].corr()

**Observation:** Correlation between app size and installs is negligible (r ≈ 0.13). File size does not meaningfully predict popularity.

## 5. Pricing Analysis

In [ ]:
plt.figure(figsize=(6,5))apps['Type'].value_counts().plot(kind='bar', color=['seagreen','salmon'])plt.title('Free vs Paid Apps')plt.xlabel('Type')plt.ylabel('Number of Apps')plt.xticks(rotation=0)plt.show()

In [ ]:
paid_apps = apps[apps['Type'] == 'Paid']plt.figure(figsize=(8,5))paid_apps['Price'].hist(bins=30, color='orange', edgecolor='black')plt.title('Price Distribution of Paid Apps')plt.xlabel('Price ($)')plt.ylabel('Number of Apps')plt.show()

In [ ]:
apps[apps['Type'] == 'Paid'].groupby('Category')['Price'].median().sort_values(ascending=False).head(10)

### Estimated Revenue by CategoryEstimated as `Price × Installs` per app, summed by category. This is a rough proxy (assumes every install was a paid purchase at full listed price) — not actual revenue, but useful for comparing relative earning potential across categories.

In [ ]:
paid_apps = paid_apps.copy()paid_apps['Estimated_Revenue'] = paid_apps['Price'] * paid_apps['Installs']revenue_by_category = paid_apps.groupby('Category')['Estimated_Revenue'].sum().sort_values(ascending=False).head(10)revenue_by_category

In [ ]:
revenue_by_category.plot(kind='bar', figsize=(10,5), color='darkgreen')plt.title('Top 10 Categories by Estimated Revenue')plt.xlabel('Category')plt.ylabel('Estimated Revenue ($)')plt.xticks(rotation=45)plt.tight_layout()plt.show()

**Observation:** Most paid apps are priced under $10, with a small number of extreme outliers (e.g. a $400 joke app). Despite `GAME` leading in installs, `FAMILY` generates the highest estimated revenue overall — driven by having both the most apps and the most paid apps of any category.

## 6. Sentiment Analysis on User ReviewsUsing the second dataset (`googleplaystore_user_reviews.csv`), we classify review sentiment with TextBlob and examine how sentiment varies by app category.

In [ ]:
reviews = pd.read_csv('googleplaystore_user_reviews.csv')reviews = reviews.dropna(subset=['Translated_Review'])print(reviews.shape)reviews.head()

In [ ]:
from textblob import TextBlobdef get_sentiment(text):    polarity = TextBlob(text).sentiment.polarity    if polarity > 0:        return 'Positive'    elif polarity < 0:        return 'Negative'    else:        return 'Neutral'reviews['My_Sentiment'] = reviews['Translated_Review'].apply(get_sentiment)reviews['My_Sentiment'].value_counts()

Sanity check: does our independently-computed sentiment match the dataset's pre-existing `Sentiment` labels?

In [ ]:
pd.crosstab(reviews['Sentiment'], reviews['My_Sentiment'])

*Result: 100% agreement — the dataset's original labels were very likely generated with the same TextBlob polarity approach. This confirms our implementation is correct, though it isn't independent external validation.*

### Sentiment by Category

In [ ]:
merged = reviews.merge(apps[['App', 'Category']], on='App', how='inner')merged.shape

In [ ]:
sentiment_by_category = merged.groupby('Category')['Sentiment'].value_counts().unstack()sentiment_pct = sentiment_by_category.div(sentiment_by_category.sum(axis=1), axis=0) * 100sentiment_pct.sort_values('Positive', ascending=False)

In [ ]:
top_categories = sentiment_pct.sort_values('Positive', ascending=False).head(15)top_categories[['Positive', 'Neutral', 'Negative']].plot(    kind='bar', stacked=True, figsize=(12,6),    color=['seagreen', 'gold', 'salmon'])plt.title('Sentiment Breakdown by Category (Top 15 by Positive %)')plt.xlabel('Category')plt.ylabel('Percentage of Reviews')plt.xticks(rotation=45, ha='right')plt.legend(title='Sentiment')plt.tight_layout()plt.show()

**Observation:** `COMICS` and `AUTO_AND_VEHICLES` have the most positive sentiment (>80%). `GAME` — despite dominating installs — has the highest negative sentiment of any category (37%) and the lowest neutral share (4.6%), suggesting games provoke unusually strong, polarized reactions compared to other categories.

## 7. Conclusion: Key Insights for App Developers1. **App volume doesn't equal app usage.** `FAMILY` has the most listed apps (1,874) on the Play Store, but `GAME` and `COMMUNICATION` dominate actual installs (13.4B and 11B respectively) — a saturated category doesn't guarantee visibility or usage.2. **App size has no meaningful impact on installs (correlation r = 0.13).** Developers don't need to obsess over minimizing app size to drive downloads — users install based on usefulness and marketing, not file size.3. **High engagement doesn't mean high satisfaction.** `GAME` is the most-installed category by far, yet it has the highest negative review sentiment (37%) of any category — suggesting that popularity and user satisfaction are not the same thing, and developers entering the GAME category should expect more polarized feedback than in steadier categories like `COMICS` or `HEALTH_AND_FITNESS`.4. **Bonus — revenue potential:** Despite `GAME` leading installs, `FAMILY` generates the highest estimated paid-app revenue overall, driven by having both the largest total app count and the most paid apps of any category.